In [ ]:
import pandas as pd

In [ ]:
pip install pandas


In [1]:
pip install pandas numpy scikit-learn matplotlib seaborn xgboost


In [2]:
import pandas as pd
dev_data=pd.read_csv('Dev_data_to_be_shared.csv')
validation_data=pd.read_csv('validation_data_to_be_shared.csv')
print(dev_data.head())
print(validation_data.head())

   account_number  bad_flag  onus_attribute_1  transaction_attribute_1  \
0               1         0               NaN                      NaN   
1               2         0          221000.0                      0.0   
2               3         0           25000.0                      0.0   
3               4         0           86000.0                      0.0   
4               5         0          215000.0                      0.0   

   transaction_attribute_2  transaction_attribute_3  transaction_attribute_4  \
0                      NaN                      NaN                      NaN   
1                      0.0                      0.0                      0.0   
2                      0.0                      0.0                      0.0   
3                      0.0                      0.0                      0.0   
4                      0.0                      0.0                      0.0   

   transaction_attribute_5  transaction_attribute_6  transaction_attribute

In [3]:
print(dev_data.info())
print(validation_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7584 entries, 0 to 7583
Columns: 1216 entries, account_number to onus_attribute_48
dtypes: float64(1212), int64(4)
memory usage: 70.4 MB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10028 entries, 0 to 10027
Columns: 1215 entries, account_number to onus_attribute_48
dtypes: float64(1214), int64(1)
memory usage: 93.0 MB
None


In [4]:
print(dev_data.isnull().sum())
print(validation_data.isnull().sum())

account_number                0
bad_flag                      0
onus_attribute_1           1922
transaction_attribute_1    1922
transaction_attribute_2    1922
                           ... 
onus_attribute_44          6702
onus_attribute_45          6702
onus_attribute_46          6702
onus_attribute_47          6702
onus_attribute_48          6702
Length: 1216, dtype: int64
account_number                0
onus_attribute_1           2637
transaction_attribute_1    2637
transaction_attribute_2    2637
transaction_attribute_3    2637
                           ... 
onus_attribute_44          8859
onus_attribute_45          8859
onus_attribute_46          8859
onus_attribute_47          8859
onus_attribute_48          8859
Length: 1215, dtype: int64


In [5]:
dev_data.fillna(dev_data.mean(),inplace=True)
validation_data.fillna(validation_data.mean(),inplace=True)

In [6]:
acc_num=validation_data['account_number']
dev_data.drop(columns=['account_number'],inplace=True)

In [7]:
print(dev_data.head())
print(validation_data.head())

   bad_flag  onus_attribute_1  transaction_attribute_1  \
0         0     155031.261039                 1.230307   
1         0     221000.000000                 0.000000   
2         0      25000.000000                 0.000000   
3         0      86000.000000                 0.000000   
4         0     215000.000000                 0.000000   

   transaction_attribute_2  transaction_attribute_3  transaction_attribute_4  \
0                  0.00159                 1.043977                114.84308   
1                  0.00000                 0.000000                  0.00000   
2                  0.00000                 0.000000                  0.00000   
3                  0.00000                 0.000000                  0.00000   
4                  0.00000                 0.000000                  0.00000   

   transaction_attribute_5  transaction_attribute_6  transaction_attribute_7  \
0                 0.006358                93.188541                 0.370894   
1         

In [8]:
print(dev_data['bad_flag'].value_counts(normalize=True))


bad_flag
0    0.984309
1    0.015691
Name: proportion, dtype: float64


In [9]:
import numpy as np
import pandas as pd

# Fill missing values with the mean of each column
dev_data.fillna(dev_data.mean(), inplace=True)

# Compute the correlation matrix and extract the upper triangle
corr_matrix = dev_data.corr()

# Efficiently find highly correlated features (greater than 0.9)
upper = np.triu(corr_matrix, k=1)
to_drop = [column for column in corr_matrix.columns if any(upper[:, corr_matrix.columns.get_loc(column)] > 0.9)]

# Drop highly correlated features
dev_data.drop(columns=to_drop, inplace=True)


In [10]:
from sklearn.model_selection import train_test_split

X = dev_data.drop(columns=['bad_flag'])
y = dev_data['bad_flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [12]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Impute missing values with the mean
imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# Initialize and fit the model
model = LogisticRegression(max_iter=500)
model.fit(X_train_scaled, y_train)

# Make predictions
y_pred = model.predict_proba(X_test_scaled)[:, 1]

# Evaluate the model
print("AUC-ROC:", roc_auc_score(y_test, y_pred))


/usr/local/lib/python3.10/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['bureau_436' 'bureau_447']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: ['bureau_436' 'bureau_447']. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(


AUC-ROC: 0.6832243695667792


In [13]:
# Drop columns with all NaN values
X_train = pd.DataFrame(X_train).dropna(axis=1, how="all")
X_test = pd.DataFrame(X_test).dropna(axis=1, how="all")


In [14]:
from sklearn.preprocessing import StandardScaler

# Scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the logistic regression model with increased iterations
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

# Predict probabilities for the test set
y_pred = model.predict_proba(X_test_scaled)[:, 1]

# Evaluate the model
auc_score = roc_auc_score(y_test, y_pred)
print("AUC-ROC:", auc_score)


AUC-ROC: 0.6832243695667792


In [15]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(eval_metric='auc')
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict_proba(X_test)[:, 1]
print("AUC-ROC:", roc_auc_score(y_test, y_pred_xgb))


AUC-ROC: 0.7250089805302105


In [19]:
from sklearn.metrics import classification_report

print("Logistic Regression Report:")
print(classification_report(y_test, model.predict(X_test_scaled)))

print("XGBoost Report:")
print(classification_report(y_test, xgb_model.predict(X_test_scaled)))


Logistic Regression Report:
              precision    recall  f1-score   support

           0       0.99      0.97      0.98      2245
           1       0.03      0.06      0.04        31

    accuracy                           0.96      2276
   macro avg       0.51      0.52      0.51      2276
weighted avg       0.97      0.96      0.96      2276

XGBoost Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      2245
           1       0.00      0.00      0.00        31

    accuracy                           0.98      2276
   macro avg       0.49      0.50      0.50      2276
weighted avg       0.97      0.98      0.98      2276



In [20]:
# Align validation data with training data columns
validation_data_cleaned_aligned = validation_data.drop(columns=['account_number']).reindex(columns=X_train.columns, fill_value=0)

# Make predictions
predictions = xgb_model.predict_proba(validation_data_cleaned_aligned)[:, 1]

print(predictions)


[1.9510903e-03 5.1826086e-05 1.4412760e-04 ... 5.9769466e-03 2.8380441e-02
 1.1450647e-05]


In [21]:
import pandas as pd

# Example data
account_numbers = [12345, 67890, 11223, 44556]  # Define your account numbers here
predictions = [0.95, 0.80, 0.72, 0.88]  # Define your predicted probabilities here

# Create the DataFrame
output = pd.DataFrame({'account_number': account_numbers, 'predicted_probability': predictions})

# Save the DataFrame to a CSV file
output.to_csv('validation_predictions.csv', index=False)
